# Notebook 02: Anatomía de Great Expectations

## Introducción

En el notebook anterior creaste tu primera validación. Ahora vamos a entender **cómo funciona Great Expectations por dentro**.

**Duración**: 30 minutos
**Nivel**: Principiante

### Objetivos de Aprendizaje:

1. Entender la jerarquía de componentes de GX
2. Dominar Context, DataSource, Asset, Batch
3. Crear Expectation Suites desde cero
4. Usar Validation Definitions
5. Generar Data Docs personalizados

## La Jerarquía de Great Expectations

Great Expectations tiene una estructura jerárquica clara:

```
Context (Contexto)
  └── DataSource (Fuente de Datos)
       └── Asset (Activo de Datos)
            └── Batch Definition (Definición de Lote)
                 └── Batch (Lote de Datos)

Expectation Suite (Suite de Expectativas)
  └── Expectation (Expectativa individual)

Validation Definition (Definición de Validación)
  = Batch Definition + Expectation Suite
  
Validation Result (Resultado de Validación)
  = Ejecutar Validation Definition
```

Vamos a explorar cada componente.

In [ ]:
import great_expectations as gx
import pandas as pd

print(f"Great Expectations versión: {gx.__version__}")

## 1. Context (Contexto)

El **Context** es el punto de entrada a Great Expectations. Gestiona:
- DataSources
- Expectation Suites
- Validation Definitions
- Data Docs

### Modos de Context:

- **Ephemeral**: En memoria, no persiste (ideal para notebooks)
- **File**: Persiste en disco (ideal para producción)

In [ ]:
# Crear contexto efímero
context = gx.get_context(mode="ephemeral")

print(" Context creado")
print(f"Tipo: {type(context)}")
print(f"\nComponentes disponibles:")
print(f"  - data_sources: {type(context.data_sources)}")
print(f"  - suites: {type(context.suites)}")
print(f"  - validation_definitions: {type(context.validation_definitions)}")

## 2. DataSource (Fuente de Datos)

Un **DataSource** representa una conexión a tus datos. Puede ser:
- Pandas DataFrame
- Base de datos SQL
- Archivos (CSV, Parquet, etc.)
- Spark DataFrame

In [ ]:
# Crear DataSource para Pandas
datasource = context.data_sources.add_pandas(name="mi_datasource")

print(" DataSource creado")
print(f"Nombre: {datasource.name}")
print(f"Tipo: {datasource.type}")

## 3. Asset (Activo de Datos)

Un **Asset** es una tabla, archivo o query específico dentro de un DataSource.

Piensa en:
- DataSource = Base de datos
- Asset = Tabla específica

In [ ]:
# Crear Asset
asset = datasource.add_dataframe_asset(name="ventas_asset")

print(" Asset creado")
print(f"Nombre: {asset.name}")
print(f"Tipo: {asset.type}")

## 4. Batch Definition (Definición de Lote)

Una **Batch Definition** define cómo obtener un lote de datos del Asset.

Ejemplos:
- Todo el DataFrame
- Datos de un mes específico
- Últimas N filas

In [ ]:
# Crear Batch Definition
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

print(" Batch Definition creado")
print(f"Nombre: {batch_def.name}")

## 5. Expectation Suite (Suite de Expectativas)

Una **Expectation Suite** es una colección de expectativas (reglas) que defines para tus datos.

Es como un "contrato de datos" que especifica qué debe cumplir tu dataset.

In [ ]:
# Crear Expectation Suite vacía
suite = context.suites.add(gx.ExpectationSuite(name="mi_suite_personalizada"))

print(" Expectation Suite creada")
print(f"Nombre: {suite.name}")
print(f"Expectativas actuales: {len(suite.expectations)}")

## 6. Expectations (Expectativas)

Las **Expectations** son las reglas individuales. Great Expectations tiene 50+ expectativas built-in.

### Categorías principales:

1. **Column Expectations**: Sobre columnas individuales
2. **Table Expectations**: Sobre la tabla completa
3. **Multi-column Expectations**: Relaciones entre columnas

In [ ]:
# Agregar expectativas a la suite

# 1. Expectativa de existencia de columna
suite.add_expectation(
    gx.expectations.ExpectColumnToExist(column="order_id")
)

# 2. Expectativa de no nulos
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")
)

# 3. Expectativa de rango
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0.01,
        max_value=10000
    )
)

# 4. Expectativa de valores permitidos
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="product_category",
        value_set=["Electronics", "Clothing", "Home", "Toys"]
    )
)

# 5. Expectativa de unicidad
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column="order_id")
)

# Guardar suite
suite.save()

print(f" Suite actualizada con {len(suite.expectations)} expectativas")
print("\nExpectativas agregadas:")
for i, exp in enumerate(suite.expectations, 1):
    print(f"  {i}. {exp.type}")

## 7. Validation Definition (Definición de Validación)

Una **Validation Definition** conecta:
- Un Batch Definition (datos)
- Una Expectation Suite (reglas)

Es la "receta" para ejecutar una validación.

In [ ]:
# Crear Validation Definition
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite,
        name="validacion_ventas"
    )
)

print(" Validation Definition creada")
print(f"Nombre: {validation_def.name}")
print(f"Batch: {validation_def.data.name}")
print(f"Suite: {validation_def.suite.name}")

## 8. Ejecutar Validación

Ahora que tenemos todos los componentes, ejecutemos la validación con datos reales.

In [ ]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Datos cargados: {len(df)} registros")
print(f"\nColumnas: {list(df.columns)}")

In [ ]:
# Ejecutar validación
resultado = validation_def.run(batch_parameters={"dataframe": df})

print("\n" + "="*60)
print("RESULTADO DE LA VALIDACIÓN")
print("="*60)
print(f"\n¿Validación exitosa?: {resultado.success}")
print(f"\nExpectativas evaluadas: {len(resultado.results)}")
print(f"Expectativas exitosas: {sum(1 for r in resultado.results if r.success)}")
print(f"Expectativas fallidas: {sum(1 for r in resultado.results if not r.success)}")

# Mostrar detalles de fallas
print("\n" + "="*60)
print("DETALLES DE EXPECTATIVAS FALLIDAS")
print("="*60)
for result in resultado.results:
    if not result.success:
        exp_type = result.expectation_config.type
        column = result.expectation_config.kwargs.get('column', 'N/A')
        unexpected_count = result.result.get('unexpected_count', 0)
        print(f"\n {exp_type}")
        print(f"   Columna: {column}")
        print(f"   Registros con problemas: {unexpected_count}")

##  Ejercicio Práctico

Ahora es tu turno. Crea una suite desde cero para validar:

1. La columna `quantity` debe existir
2. La columna `quantity` no debe tener nulos
3. La columna `quantity` debe estar entre 1 y 100
4. La columna `order_date` debe existir
5. La columna `order_date` no debe tener nulos

### Pistas:
- Usa `ExpectColumnToExist`
- Usa `ExpectColumnValuesToNotBeNull`
- Usa `ExpectColumnValuesToBeBetween`

In [ ]:
# TU CÓDIGO AQUÍ
# Crea una nueva suite llamada "ejercicio_suite"

ejercicio_suite = context.suites.add(gx.ExpectationSuite(name="ejercicio_suite"))

# Agrega las 5 expectativas
# 1. ...
# 2. ...
# 3. ...
# 4. ...
# 5. ...

ejercicio_suite.save()
print(f"Suite creada con {len(ejercicio_suite.expectations)} expectativas")

In [ ]:
# SOLUCIÓN (descomenta para ver)

# ejercicio_suite.add_expectation(
#     gx.expectations.ExpectColumnToExist(column="quantity")
# )
# ejercicio_suite.add_expectation(
#     gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity")
# )
# ejercicio_suite.add_expectation(
#     gx.expectations.ExpectColumnValuesToBeBetween(
#         column="quantity",
#         min_value=1,
#         max_value=100
#     )
# )
# ejercicio_suite.add_expectation(
#     gx.expectations.ExpectColumnToExist(column="order_date")
# )
# ejercicio_suite.add_expectation(
#     gx.expectations.ExpectColumnValuesToNotBeNull(column="order_date")
# )
# ejercicio_suite.save()

# # Validar
# ejercicio_val_def = context.validation_definitions.add(
#     gx.ValidationDefinition(
#         data=batch_def,
#         suite=ejercicio_suite,
#         name="validacion_ejercicio"
#     )
# )
# resultado_ejercicio = ejercicio_val_def.run(batch_parameters={"dataframe": df})
# print(f"\n¿Validación exitosa?: {resultado_ejercicio.success}")

## Generar Data Docs

Finalmente, generemos la documentación visual de todas nuestras validaciones.

In [ ]:
# Generar Data Docs
context.build_data_docs()

print("\n" + "="*60)
print(" Data Docs generados exitosamente!")
print("="*60)
print("\nEn el Data Doc podrás ver:")
print("  - Todas las suites creadas")
print("  - Resultados de validaciones")
print("  - Estadísticas detalladas")
print("  - Gráficos interactivos")
print("\nAbriendo en tu navegador...")

context.open_data_docs()

##  Resumen del Notebook

Has aprendido la anatomía completa de Great Expectations:

### Componentes Principales:

1. **Context**: Punto de entrada, gestiona todo
2. **DataSource**: Conexión a tus datos
3. **Asset**: Tabla/archivo específico
4. **Batch Definition**: Cómo obtener un lote de datos
5. **Expectation Suite**: Colección de reglas
6. **Expectation**: Regla individual
7. **Validation Definition**: Batch + Suite
8. **Validation Result**: Resultado de ejecutar validación

### Flujo de Trabajo:

```
1. Crear Context
2. Configurar DataSource → Asset → Batch Definition
3. Crear Expectation Suite
4. Agregar Expectations
5. Crear Validation Definition
6. Ejecutar validación
7. Generar Data Docs
```

